# Danh gia truy xuat website voi BM25

In [1]:
import os
import nltk
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from whoosh.index import create_in
from whoosh.fields import *
from whoosh.analysis import StandardAnalyzer
from whoosh import qparser
from whoosh import scoring
import whoosh.index as index

import pytrec_eval
import math


nltk.download('punkt_tab')
nltk.download('stopwords')
stoplist = stopwords.words("english")
stoplist.append('oh')
puncts = ['.', ',', ':', '`', '"', "'", '!', '?', "``", "''"]
ps = PorterStemmer()
import shutil

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mt200\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
import os
import re
import pandas as pd
import numpy as np
import networkx as nx
from tqdm import tqdm
from pyvi import ViTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

In [3]:
from nltk.tokenize import sent_tokenize

def split_sentences(text):
    return sent_tokenize(text)

In [4]:
from nltk.tokenize import sent_tokenize
from pyvi import ViTokenizer

def tokenize_vi_sentence_level(text: str) -> list[str]:
    sentences = sent_tokenize(text)
    tokens = []

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue

        sent_tokens = ViTokenizer.tokenize(sent)
        tokens.extend(sent_tokens.split())

    return tokens


In [5]:
import re

VI_TOKEN_REGEX = re.compile(
    r"[a-zàáạảãâầấậẩẫăằắặẳẵ"
    r"èéẹẻẽêềếệểễ"
    r"ìíịỉĩ"
    r"òóọỏõôồốộổỗơờớợởỡ"
    r"ùúụủũưừứựửữ"
    r"ỳýỵỷỹđ0-9_]+$"
)

def is_valid_vi_token(token: str) -> bool:
    return bool(VI_TOKEN_REGEX.fullmatch(token))


In [6]:
def load_stopwords(path):
    with open(path, "r", encoding="utf-8") as f:
        stopwords = set(
            line.strip().lower()
            for line in f
            if line.strip()
        )
    return stopwords

STOPWORDS_PATH = "../stopword/vietnamese-stopwords-dash.txt"
vi_stopwords = load_stopwords(STOPWORDS_PATH)

print(f"🛑 Đã load {len(vi_stopwords)} stopword")

🛑 Đã load 1942 stopword


In [7]:
# === Đọc dữ liệu và tiền xử lý ===
import re
import unicodedata

def clean_text(text):
    text = unicodedata.normalize("NFC", text)
    
    # Xóa URL
    text = re.sub(r"http\S+|www\S+", "", text)

    # text = text.lower()

    # # Loại ký tự không cần thiết (giữ chữ, số, dấu câu cơ bản)
    # text = re.sub(r"[^0-9a-zàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễ"
    #               r"ìíịỉĩòóọỏõôồốộổỗơờớợởỡ"
    #               r"ùúụủũưừứựửữỳýỵỷỹđ\s.,!?]", " ", text)

    # Chuẩn hóa dấu câu
    text = re.sub(r"[.,!?]+", " ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [8]:
def preprocess_query(
    query: str,
    stopwords: set[str] | None = None
) -> list[str]:
    """
    Input : raw query string
    Output: list[token] đã clean + tokenize + remove stopword
    """

    # 1. Clean
    query = clean_text(query)

    # 2. Tokenize
    tokens = tokenize_vi_sentence_level(query)

    # 3. Normalize + filter
    processed_tokens = []
    for tok in tokens:
        tok = tok.lower()

        if not is_valid_vi_token(tok):
            continue

        if tok.isnumeric():
            continue

        if stopwords and tok in stopwords:
            continue

        processed_tokens.append(tok)

    return " ".join(processed_tokens)


In [9]:
STOPWORDS_PATH = "../stopword/vietnamese-stopwords-dash.txt"
vi_stopwords = load_stopwords(STOPWORDS_PATH)

query = "Những địa điểm du lịch nổi tiếng nhất ở Hà Nội là gì?"

tokens = preprocess_query(query, vi_stopwords)

print(tokens)

địa_điểm du_lịch nổi_tiếng hà_nội


In [10]:
import re

def preprocess(tokens):
    """
    tokens: list[str] đã được lọc term
    return: string dùng cho indexing
    """
    tokens = [t.lower() for t in tokens if len(t) > 1]
    return " ".join(tokens)


In [11]:
from whoosh.fields import Schema, TEXT, ID
from whoosh.analysis import StandardAnalyzer

from whoosh.analysis import KeywordAnalyzer

def create_schema():
    # Sử dụng KeywordAnalyzer vì bạn đã preprocess thủ công rồi
    # analyzer này sẽ giữ nguyên các token bạn đã tách
    my_analyzer = KeywordAnalyzer(lowercase=False) 
    return Schema(
        docid=ID(stored=True, unique=True),
        title=TEXT(stored=True, analyzer=my_analyzer),
        content=TEXT(stored=True, analyzer=my_analyzer)
    )


In [12]:
import shutil
import os

def build_index(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = preprocess_query(row["title"])
        doc_file = row["document"]
        # print(f"Indexing docid={docid}, file={doc_file}")
        tokens = doc_terms.get(doc_file, [])
        # print(f"Số token: {len(tokens)}")
        if not tokens:
            continue

        content = preprocess(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix

In [13]:
def readQuery(query_csv):
    df = pd.read_csv(query_csv)
    queries = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        queries[qid] = preprocess_query(row["query"], vi_stopwords)

    return queries

In [14]:
import ast

def readGroundTruth(query_csv, meta_csv):
    meta = pd.read_csv(meta_csv)
    url2docid = dict(zip(meta["url"], meta["id"]))

    df = pd.read_csv(query_csv)
    qrels = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        qrels[qid] = {}

        urls = ast.literal_eval(row["urls"])
        for u in urls:
            if u in url2docid:
                qrels[qid][str(url2docid[u])] = 1

    return qrels


In [15]:
import pytrec_eval
import math

def print_best_worst_queries(
    RunResults,
    Queries,
    GroundTruth,
    top_k=5,
    retrieved_k=10
):
    evaluator = pytrec_eval.RelevanceEvaluator(
        GroundTruth,
        {"map", "P_10"}
    )
    results = evaluator.evaluate(RunResults)

    # Lọc query hợp lệ
    valid = [
        (qid, res["map"], res["P_10"])
        for qid, res in results.items()
        if not math.isnan(res["map"])
    ]

    # Sort theo MAP (bạn có thể đổi sang P@10 nếu muốn)
    valid_sorted = sorted(valid, key=lambda x: x[1], reverse=True)

    best = valid_sorted[:top_k]
    worst = valid_sorted[-top_k:]

    def print_block(title, items):
        print("\n" + "=" * 60)
        print(title)
        print("=" * 60)

        for qid, map_score, p10_score in items:
            print(f"\nQuery ID : {qid}")
            print(f"Query    : {Queries[qid]}")
            print(f"MAP      : {map_score:.4f}")
            print(f"P@10     : {p10_score:.4f}")

            # Relevant docs
            rel_docs = [
                docid for docid, rel in GroundTruth[qid].items()
                if rel > 0
            ]
            print(f"Relevant docs ({len(rel_docs)}): {rel_docs[:retrieved_k]}")

            # Retrieved docs
            retrieved = list(RunResults[qid].keys())[:retrieved_k]
            print(f"Top retrieved docs: {retrieved}")

    print_block("🔥 TOP QUERIES (Highest MAP)", best)
    print_block("❄️ WORST QUERIES (Lowest MAP)", worst)


In [ ]:
from whoosh.qparser import MultifieldParser
from whoosh.qparser import QueryParser
from whoosh.scoring import BM25F

def bm25_search(ix, query, top_k=100):
    results = {}
    with ix.searcher(weighting=BM25F()) as searcher:
        og = qparser.OrGroup.factory(0.9) # Cho phép OR nhưng ưu tiên các doc chứa nhiều từ hơn
        parser = MultifieldParser(["title", "content"], ix.schema, group=og)
        parser.add_plugin(qparser.FuzzyTermPlugin()) # Có thể thêm tìm kiếm mờ nếu cần
        parser = QueryParser("content", ix.schema, group=qparser.OrGroup)
        q = parser.parse(query)

        hits = searcher.search(q, limit=top_k)

        for hit in hits:
            results[str(hit["docid"])] = float(hit.score)

    return results


In [17]:
def run_bm25_all_queries(ix, queries, top_k=100):
    run = {}

    for qid, query in queries.items():
        run[qid] = bm25_search(ix, query, top_k)

    return run


In [18]:
def evaluate_set_retrieval_at_k(GroundTruth, RunResults, cutoffs=[5,10,20]):
    """
    GroundTruth: dict {qid: {docid: relevance}}
    RunResults : dict {qid: {docid: score}}
    """

    per_query = {}
    avg_metrics = {k: {"P":0, "R":0, "F1":0} for k in cutoffs}
    n = len(GroundTruth)

    print("========== Per-query results ==========")

    for qid in GroundTruth:
        relevant = set(GroundTruth[qid].keys())

        ranked_docs = sorted(
            RunResults.get(qid, {}).items(),
            key=lambda x: x[1],
            reverse=True
        )

        per_query[qid] = {}

        for k in cutoffs:
            retrieved_k = set(docid for docid, _ in ranked_docs[:k])

            tp = len(relevant & retrieved_k)
            fp = len(retrieved_k) - tp
            fn = len(relevant) - tp

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

            per_query[qid][f"P@{k}"] = precision
            per_query[qid][f"R@{k}"] = recall
            per_query[qid][f"F1@{k}"] = f1

            avg_metrics[k]["P"] += precision
            avg_metrics[k]["R"] += recall
            avg_metrics[k]["F1"] += f1

        print(f"Query {qid}")
        for k in cutoffs:
            print(f"  @ {k}")
            print(f"    Precision : {per_query[qid][f'P@{k}']:.4f}")
            print(f"    Recall    : {per_query[qid][f'R@{k}']:.4f}")
            print(f"    F1        : {per_query[qid][f'F1@{k}']:.4f}")
        print("-" * 30)

    print("\n========== Average over all queries ==========")
    for k in cutoffs:
        print(f"@{k}")
        print(f"  Precision : {avg_metrics[k]['P']/n:.4f}")
        print(f"  Recall    : {avg_metrics[k]['R']/n:.4f}")
        print(f"  F1        : {avg_metrics[k]['F1']/n:.4f}")

    return per_query, avg_metrics


## Bo cac tu it xuat hien

In [19]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
run = run_bm25_all_queries(ix, queries, top_k=100)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


In [20]:
queries

{'1': 'nhà_thờ kon_tum',
 '2': 'vũng_tàu địa_điểm đẹp',
 '3': 'địa_điểm du_lịch nổi_tiếng hà_nội',
 '4': 'đi du_lịch đà_nẵng',
 '5': 'du_lịch hội_an trải nghiệm đặc_sắc',
 '6': 'lý_tưởng du_lịch sa_pa',
 '7': 'tham_quan lỡ huế',
 '8': 'du_lịch ninh_bình đi tràng_an tam_cốc',
 '9': 'địa_điểm du_lịch sinh_thái nổi_bật miền tây_nam_bộ',
 '10': 'check in đẹp đà_lạt giới trẻ',
 '11': 'du_lịch hạ_long tour hoạt_động hấp_dẫn',
 '12': 'địa_điểm du_lịch tâm_linh nổi_tiếng việt_nam',
 '13': 'đi du_lịch côn_đảo mùa',
 '14': 'địa_điểm du_lịch tp hcm đi tuần',
 '15': 'du_lịch mộc_châu hấp_dẫn mùa hoa',
 '16': 'vườn quốc_gia đẹp nổi_tiếng việt_nam',
 '17': 'du_lịch quy_nhơn bãi biển hoang_sơ',
 '18': 'du_lịch miền huế đi',
 '19': 'đồng_nai núi',
 '20': 'chùa nổi_tiếng hà_nội',
 '21': 'núi nổi_tiếng leo sa_pa lào_cai',
 '22': 'thác đẹp nổi_tiếng đà_lạt',
 '23': 'hang_động nổi_tiếng quảng_bình',
 '24': 'di_tích lịch_sử nổi_bật cố_đô huế',
 '25': 'cầu nổi_tiếng check in đà_nẵng',
 '26': 'làng_nghề truy

In [83]:
print_best_worst_queries(run, queries, qrels, top_k=5, retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 19
Query    : đồng_nai núi
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['30', '218']
Top retrieved docs: ['218', '30', '376', '377', '162', '392', '216', '149', '168', '232']

Query ID : 37
Query    : thành nhà_hồ
MAP      : 1.0000
P@10     : 0.1000
Relevant docs (1): ['151']
Top retrieved docs: ['151', '155', '339', '150', '128', '291', '89', '104', '319', '178']

Query ID : 39
Query    : núi việt_nam
MAP      : 1.0000
P@10     : 0.1000
Relevant docs (1): ['138']
Top retrieved docs: ['138', '160', '186', '253', '237', '134', '163', '218', '140', '164']

Query ID : 1
Query    : nhà_thờ kon_tum
MAP      : 0.8333
P@10     : 0.2000
Relevant docs (2): ['494', '495']
Top retrieved docs: ['494', '57', '495', '172', '201', '205', '207', '232', '214', '216']

Query ID : 33
Query    : đèo đẹp nổi_tiếng phượt miền trung
MAP      : 0.7500
P@10     : 0.2000
Relevant docs (2): ['436', '437']
Top retrieved docs: ['437', '37', '9', '436', '111', '30

In [84]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
------------------------------
Query 4
  @ 5
    Precision : 0.8000
    Recall    : 0.3333
    F1        : 0.4706
  @ 10
    Precision : 0.5000
    Recall    : 0.4167
    F1        : 0.4545
  @

## Khong bo least

In [85]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
run = run_bm25_all_queries(ix, queries, top_k=100)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


In [86]:
run

{'1': {'494': 11.45184427236099,
  '57': 10.648204368104288,
  '495': 9.462169493994441,
  '172': 9.155025676899282,
  '201': 9.100405748969266,
  '205': 8.970613667356341,
  '207': 8.832891106451203,
  '232': 8.531846692882567,
  '214': 8.50111918244803,
  '216': 8.50111918244803,
  '297': 8.01747516263861,
  '381': 7.964629241463779,
  '405': 7.845898777871012,
  '368': 7.7931747445774775,
  '221': 7.511154013152378,
  '224': 6.660989757738323,
  '54': 6.378819047192888,
  '160': 6.19373084911831,
  '313': 6.032755772391146,
  '67': 5.799417135586013,
  '391': 5.746238987988986,
  '328': 5.555011281393305,
  '143': 5.539223729049308,
  '410': 5.53893659377319,
  '162': 5.512491857837366,
  '372': 5.466970656984692,
  '333': 5.426462737784659,
  '349': 5.361518442994279,
  '400': 5.333166855691026,
  '331': 5.266907924032326,
  '82': 5.230610364321068,
  '363': 5.210273434991198,
  '362': 5.210114931850449,
  '186': 5.20797915433721,
  '150': 5.181499725260151,
  '314': 5.170503421130

In [87]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
------------------------------
Query 4
  @ 5
    Precision : 0.8000
    Recall    : 0.3333
    F1        : 0.4706
  @ 10
    Precision : 0.5000
    Recall    : 0.4167
    F1        : 0.4545
  @

## Them title

In [26]:
import shutil
import os

def build_index_title(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = preprocess_query(row["title"])
        doc_file = row["document"]

        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        content = preprocess(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [ ]:
from whoosh import qparser
from whoosh.qparser import MultifieldParser
from whoosh.scoring import BM25F

def bm25_search_title(ix, query_str, top_k=100, B=0.75, K1=1.2, title=1.5, content=1.0):
    results = {}
    
    # Cấu hình BM25F: b=0.75 là mặc định, có thể tinh chỉnh sau
    # title_B=0.75, content_B=0.75
    weighting = BM25F(B=B, K1=K1)

    with ix.searcher(weighting=weighting) as searcher:
        # 1. Định nghĩa trọng số cho từng trường (Title quan trọng hơn Content)
        # Ở đây ta ưu tiên Title gấp 2 lần Content
        field_boosts = {
            "title": 1.5,
            "content": 1.0
        }

        # 2. Sử dụng OrGroup để tránh việc query quá dài dẫn đến 0 kết quả
        # Những tài liệu chứa nhiều từ khóa hơn vẫn sẽ đứng đầu nhờ thuật toán BM25
        parser = MultifieldParser(
            ["title", "content"],
            schema=ix.schema,
            fieldboosts=field_boosts,
            group=qparser.OrGroup # Chuyển từ AND sang OR
        )

        # Parse câu truy vấn
        q = parser.parse(query_str)
        
        # In ra để debug xem Whoosh thực sự tìm cái gì (Optional)
        # print(f"Query thực tế: {q}")

        hits = searcher.search(q, limit=top_k)

        for hit in hits:
            results[str(hit["docid"])] = float(hit.score)

    return results

In [ ]:
def run_bm25_all_queries_title(ix, queries, top_k=50, B=0.75, K1=1.2, title=1.5, content=1.0):
    run = {}

    for qid, query in queries.items():
        run[qid] = bm25_search_title(ix, query, top_k, B, K1, title, content)
    return run


In [ ]:
def compute_MAP(GroundTruth, RunResults):
    evaluator = pytrec_eval.RelevanceEvaluator(
        GroundTruth, {"map"}
    )
    results = evaluator.evaluate(RunResults)

    MAP = 0.0
    valid_queries = 0

    for qid, res in results.items():
        if not math.isnan(res["map"]):
            MAP += res["map"]
            valid_queries += 1

    return MAP / valid_queries if valid_queries > 0 else 0.0


In [ ]:
import optuna

In [ ]:
def objective(trial):
    # 🔧 Siêu tham số cần tối ưu
    top_k = trial.suggest_int("top_k", 20, 100)
    B = trial.suggest_int("B", 5, 50)
    K1 = trial.suggest_float("K1", 0.0, 1.0)
    content = trial.suggest_float("content", 0.5, 2.0)
    title = trial.suggest_float("title", 0.5, 2.0)

    RunResults = {}

    for qid, qtext in queries.items():
        bm25_res = RunResults[qid]

        fused = run_bm25_all_queries_title(
            ix, 
            qtext, 
            top_k=50,
            B=0.75,
            K1=1.2,
            title=1.5,
            content=1.0
        )

        # ⚠️ pytrec_eval yêu cầu score là float
        RunResults[qid] = {
            docid: float(score)
            for docid, score in fused.items()
        }

    # 🎯 Objective = MAP
    return compute_MAP(qrels, RunResults)

In [28]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index_title(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
all_results = {}

for qid, query_text in queries.items():
    all_results[qid] = bm25_search_title(ix, query_text, top_k=50)

print("✅ Đã chạy xong tất cả truy vấn")

✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


In [49]:
queries

{'1': 'nhà_thờ kon_tum',
 '2': 'vũng_tàu địa_điểm đẹp',
 '3': 'địa_điểm du_lịch nổi_tiếng hà_nội',
 '4': 'đi du_lịch đà_nẵng',
 '5': 'du_lịch hội_an trải nghiệm đặc_sắc',
 '6': 'lý_tưởng du_lịch sa_pa',
 '7': 'tham_quan lỡ huế',
 '8': 'du_lịch ninh_bình đi tràng_an tam_cốc',
 '9': 'địa_điểm du_lịch sinh_thái nổi_bật miền tây_nam_bộ',
 '10': 'check in đẹp đà_lạt giới trẻ',
 '11': 'du_lịch hạ_long tour hoạt_động hấp_dẫn',
 '12': 'địa_điểm du_lịch tâm_linh nổi_tiếng việt_nam',
 '13': 'đi du_lịch côn_đảo mùa',
 '14': 'địa_điểm du_lịch tp hcm đi tuần',
 '15': 'du_lịch mộc_châu hấp_dẫn mùa hoa',
 '16': 'vườn quốc_gia đẹp nổi_tiếng việt_nam',
 '17': 'du_lịch quy_nhơn bãi biển hoang_sơ',
 '18': 'du_lịch miền huế đi',
 '19': 'đồng_nai núi',
 '20': 'chùa nổi_tiếng hà_nội',
 '21': 'núi nổi_tiếng leo sa_pa lào_cai',
 '22': 'thác đẹp nổi_tiếng đà_lạt',
 '23': 'hang_động nổi_tiếng quảng_bình',
 '24': 'di_tích lịch_sử nổi_bật cố_đô huế',
 '25': 'cầu nổi_tiếng check in đà_nẵng',
 '26': 'làng_nghề truy

In [29]:
print_best_worst_queries(all_results, queries, qrels, top_k=5, retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 20
Query    : chùa nổi_tiếng hà_nội
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['350', '49']
Top retrieved docs: ['350', '49', '18', '444', '17', '403', '445', '366', '389', '390']

Query ID : 30
Query    : ruộng bậc_thang nổi_tiếng miền núi bắc
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['308', '309']
Top retrieved docs: ['308', '309', '188', '86', '62', '111', '126', '70', '100', '464']

Query ID : 33
Query    : đèo đẹp nổi_tiếng phượt miền trung
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['436', '437']
Top retrieved docs: ['436', '437', '343', '490', '188', '37', '381', '9', '439', '111']

Query ID : 37
Query    : thành nhà_hồ
MAP      : 1.0000
P@10     : 0.1000
Relevant docs (1): ['151']
Top retrieved docs: ['151', '338', '339', '155', '150', '128', '291', '89', '104', '319']

Query ID : 38
Query    : nhà_thờ nổi_tiếng chụp ảnh đà_lạt
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['362', '363']
To

In [ ]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    all_results
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.4000
    Recall    : 0.2500
    F1        : 0.3077
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
------------------------------
Query 4
  @ 5
    Precision : 0.8000
    Recall    : 0.3333
    F1        : 0.4706
  @ 10
    Precision : 0.4000
    Recall    : 0.3333
    F1        : 0.3636
  @

## Tach tu

In [20]:
def preprocess_query_word(
    query: str,
    stopwords: set[str] | None = None
) -> list[str]:
    """
    Input : raw query string
    Output: list[token] đã clean + tokenize + remove stopword
    """

    # 1. Clean
    query = clean_text(query)

    # 2. Tokenize
    tokens = tokenize_vi_sentence_level(query)

    # 3. Normalize + filter
    processed_tokens = []
    for tok in tokens:
        tok = tok.lower()

        if not is_valid_vi_token(tok):
            continue

        if tok.isnumeric():
            continue

        if stopwords and tok in stopwords:
            continue

        processed_tokens.append(tok)

    return " ".join(processed_tokens)


In [21]:
import re

def preprocess_word(tokens):
    """
    tokens: list[str] đã được lọc term
    return: string dùng cho indexing
    """
    processed = []

    for t in tokens:
        t = t.lower()
        if len(t) <= 1:
            continue

        if "_" in t:
            processed.extend(t.split("_"))
        else:
            processed.append(t)

    return " ".join(processed)


In [22]:
import shutil
import os

def build_index_word(index_dir, meta_csv, json_path):
    from whoosh.index import create_in
    from whoosh.fields import Schema

    # 🔥 Xóa index cũ nếu tồn tại
    if os.path.exists(index_dir):
        shutil.rmtree(index_dir)

    os.mkdir(index_dir)

    schema = create_schema()
    ix = create_in(index_dir, schema)
    writer = ix.writer()

    import json
    import pandas as pd

    df = pd.read_csv(meta_csv)

    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    for _, row in df.iterrows():
        docid = str(row["id"])
        title = preprocess_query_word(row["title"])
        doc_file = row["document"]

        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        content = preprocess_word(tokens)

        writer.add_document(
            docid=docid,
            title=title,
            content=content
        )

    writer.commit()
    return ix


In [23]:
def readQuery_word(query_csv):
    df = pd.read_csv(query_csv)
    queries = {}

    for _, row in df.iterrows():
        qid = str(row["stt"])
        queries[qid] = preprocess_query(row["query"])

    return queries


In [30]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index_word(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery_word(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))



✅ Đã xây dựng xong index
Số truy vấn: 40


In [31]:
# 3. BM25
all_run = {}
for qid, query_text in queries.items():
    all_run[qid] = bm25_search_title(ix, query_text, top_k=50)
print("✅ Đã chạy xong tất cả truy vấn")

✅ Đã chạy xong tất cả truy vấn


In [ ]:
run

In [32]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
------------------------------
Query 4
  @ 5
    Precision : 0.8000
    Recall    : 0.3333
    F1        : 0.4706
  @ 10
    Precision : 0.5000
    Recall    : 0.4167
    F1        : 0.4545
  @

## BM25+KNN

In [33]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop_least.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# 1. Index
ix = build_index_title(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. BM25
run = run_bm25_all_queries_title(ix, queries, top_k=50)
print("✅ Đã chạy xong tất cả truy vấn")


✅ Đã xây dựng xong index
Số truy vấn: 40
✅ Đã chạy xong tất cả truy vấn


### KNN

In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [35]:
def build_tfidf_vectors(meta_csv, json_path):
    import pandas as pd
    import json

    df = pd.read_csv(meta_csv)
    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    docids = []
    corpus = []

    for _, row in df.iterrows():
        docid = str(row["id"])
        doc_file = row["document"]
        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        text = preprocess(tokens)
        docids.append(docid)
        corpus.append(text)

    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        lowercase=False,
        norm="l2"
    )

    X = vectorizer.fit_transform(corpus)

    return docids, X, vectorizer


In [36]:
from sklearn.metrics.pairwise import cosine_similarity

def knn_search(query, vectorizer, X_docs, docids, top_k=100):
    q_vec = vectorizer.transform([query])
    sims = cosine_similarity(q_vec, X_docs)[0]

    top_idx = np.argsort(sims)[::-1][:top_k]

    results = {}
    for i in top_idx:
        if sims[i] > 0:
            results[docids[i]] = float(sims[i])

    return results


In [37]:
def run_knn_all_queries(queries, vectorizer, X_docs, docids, top_k=100):
    run = {}
    for qid, query in queries.items():
        run[qid] = knn_search(
            query, vectorizer, X_docs, docids, top_k
        )
    return run


In [38]:
def normalize_scores(run):
    norm_run = {}
    for qid, docs in run.items():
        if not docs:
            norm_run[qid] = {}
            continue
        scores = list(docs.values())
        min_s, max_s = min(scores), max(scores)
        norm_run[qid] = {}
        for d, s in docs.items():
            if max_s > min_s:
                norm_run[qid][d] = (s - min_s) / (max_s - min_s)
            else:
                norm_run[qid][d] = 0.0
    return norm_run


In [39]:
def fuse_runs(bm25, knn, alpha=0.6):
    bm25 = normalize_scores(bm25)
    knn = normalize_scores(knn)

    FUSED = {}

    for qid in bm25:
        docs = set(bm25[qid].keys()) | set(knn.get(qid, {}).keys())
        fused_scores = {}
        for d in docs:
            fused_scores[d] = (
                alpha * bm25[qid].get(d, 0) +
                (1 - alpha) * knn.get(qid, {}).get(d, 0)
            )
        FUSED[qid] = fused_scores

    return FUSED


In [40]:
# 1. BM25
RunResults_BM25 = run_bm25_all_queries_title(ix, queries, top_k=100)
print("✅ Đã chạy xong tất cả truy vấn BM25")
# 2. KNN
docids, X_docs, vectorizer = build_tfidf_vectors(META_CSV, JSON_DIR)
RunResults_KNN = run_knn_all_queries(
    queries, vectorizer, X_docs, docids, top_k=100
)
print("✅ Đã chạy xong tất cả truy vấn KNN")
# 3. Fusion
RunResults_Fused = fuse_runs(
    RunResults_BM25,
    RunResults_KNN,
    alpha=0.6
)
print("✅ Đã fuse xong kết quả BM25 và KNN")

✅ Đã chạy xong tất cả truy vấn BM25


c:\Users\mt200\OneDrive\Desktop\AI\InformationRetrieval\Project_InformationRetrieval\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


✅ Đã chạy xong tất cả truy vấn KNN
✅ Đã fuse xong kết quả BM25 và KNN


In [41]:
print_best_worst_queries(RunResults_Fused, queries, qrels, top_k=5, retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 11
Query    : du_lịch hạ_long tour hoạt_động hấp_dẫn
MAP      : 1.0000
P@10     : 0.4000
Relevant docs (4): ['50', '122', '132', '133']
Top retrieved docs: ['95', '226', '319', '168', '339', '371', '58', '289', '442', '472']

Query ID : 19
Query    : đồng_nai núi
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['30', '218']
Top retrieved docs: ['190', '108', '366', '216', '168', '200', '289', '230', '124', '396']

Query ID : 20
Query    : chùa nổi_tiếng hà_nội
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['350', '49']
Top retrieved docs: ['108', '95', '366', '216', '466', '443', '289', '124', '288', '205']

Query ID : 27
Query    : chợ đêm nổi_tiếng đông đà_lạt
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['143', '38']
Top retrieved docs: ['95', '319', '366', '216', '339', '486', '371', '442', '472', '320']

Query ID : 30
Query    : ruộng bậc_thang nổi_tiếng miền núi bắc
MAP      : 1.0000
P@10     : 0.2000
Relevant doc

In [42]:
RunResults_Fused

{'1': {'190': 0.00881478479156547,
  '95': 0.04077760547696092,
  '319': 0.023795274732915993,
  '216': 0.16593397171885227,
  '443': 0.045900093461802675,
  '124': 0.02087610134269637,
  '265': 0.03441075328151712,
  '205': 0.193930206275185,
  '123': 0.015687805880332587,
  '57': 0.6540175603262324,
  '337': 0.018896501532191072,
  '397': 0.028359143594903928,
  '173': 0.0,
  '347': 0.03225764456312764,
  '445': 0.05233138742024356,
  '167': 0.06806020433056306,
  '5': 0.05087246353797794,
  '115': 0.011647732251951688,
  '391': 0.07122781619873726,
  '37': 0.02944253541841342,
  '172': 0.1926873702088006,
  '158': 0.011916366019910635,
  '368': 0.14430339179980342,
  '186': 0.0576583150374515,
  '237': 0.04582035010993131,
  '33': 0.04257288512599759,
  '14': 0.009020234465850109,
  '362': 0.39815334451424317,
  '143': 0.06693649238420342,
  '464': 0.05563472635164931,
  '357': 4.1849308782477816e-05,
  '151': 0.022305045275216134,
  '90': 0.0009891047514056304,
  '405': 0.148278298

In [44]:
print_best_worst_queries(RunResults_Fused, queries, qrels, top_k=5, retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 11
Query    : du_lịch hạ_long tour hoạt_động hấp_dẫn
MAP      : 1.0000
P@10     : 0.4000
Relevant docs (4): ['50', '122', '132', '133']
Top retrieved docs: ['95', '226', '319', '168', '339', '371', '58', '289', '442', '472']

Query ID : 19
Query    : đồng_nai núi
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['30', '218']
Top retrieved docs: ['190', '108', '366', '216', '168', '200', '289', '230', '124', '396']

Query ID : 20
Query    : chùa nổi_tiếng hà_nội
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['350', '49']
Top retrieved docs: ['108', '95', '366', '216', '466', '443', '289', '124', '288', '205']

Query ID : 27
Query    : chợ đêm nổi_tiếng đông đà_lạt
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['143', '38']
Top retrieved docs: ['95', '319', '366', '216', '339', '486', '371', '442', '472', '320']

Query ID : 30
Query    : ruộng bậc_thang nổi_tiếng miền núi bắc
MAP      : 1.0000
P@10     : 0.2000
Relevant doc

In [43]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    RunResults_Fused
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.3000
    Recall    : 0.3750
    F1        : 0.3333
  @ 20
    Precision : 0.1500
    Recall    : 0.3750
    F1        : 0.2143
------------------------------
Query 4
  @ 5
    Precision : 0.6000
    Recall    : 0.2500
    F1        : 0.3529
  @ 10
    Precision : 0.7000
    Recall    : 0.5833
    F1        : 0.6364
  @

## BM25 + Phan hoi

In [21]:
from whoosh import index, scoring, qparser

from whoosh import index, scoring, qparser
from whoosh.qparser import MultifieldParser

def bm25_retrieve(ix_dir, queries, k1=1.2, b=0.75, top_k=100):
    idx = index.open_dir(ix_dir)

    weighting = scoring.BM25F(
        K1=k1,
        B=b
    )

    searcher = idx.searcher(weighting=weighting)

    parser = MultifieldParser(
        ["title", "content"],
        schema=idx.schema,
        fieldboosts={
            "title": 1.5,
            "content": 1.0
        },
        group=qparser.OrGroup
    )

    runs = {}
    for qid, qtext in queries.items():
        q = parser.parse(qtext)
        results = searcher.search(q, limit=top_k)

        # 🔥 Lưu docnum để dùng cho PRF
        runs[qid] = [hit.docnum for hit in results]

    return runs, searcher

In [22]:
def get_pseudo_relevant(runs, K=10):
    RD = {}
    for qid in runs:
        RD[qid] = runs[qid][:K]
    return RD

In [23]:
from collections import Counter

def collect_term_stats(searcher, RD):
    term_df = Counter()
    R = 0

    for qid in RD:
        for docnum in RD[qid]:
            doc = searcher.stored_fields(docnum)
            if "content" not in doc:
                continue

            terms = set(doc["content"].split())
            for t in terms:
                term_df[t] += 1
            R += 1

    return term_df, R


In [24]:
def estimate_p(term_df, R, K_smooth=0.75, p_prior=0.5):
    p = {}
    for term, df in term_df.items():
        p[term] = (df + K_smooth * p_prior) / (R + K_smooth)
    return p


In [25]:
import math

def compute_idf(searcher, term, field="content"):
    N = searcher.doc_count()
    df = searcher.doc_frequency(field, term)
    return math.log((N - df + 0.5) / (df + 0.5))


In [26]:
import math

def compute_weights(searcher, p_terms):
    weights = {}

    for term, p in p_terms.items():
        if 0 < p < 1:
            idf = compute_idf(searcher, term, field="content")
            weights[term] = idf + math.log(p / (1 - p))

    return weights


In [27]:
def expand_query_safe(original_query, weights, top_m=5, expansion_weight_scale=0.5):
    """
    original_query: Chuỗi văn bản thô (chưa có boost)
    weights: Thống kê trọng số từ PRF
    expansion_weight_scale: Hệ số điều chỉnh độ tin cậy của từ mới (0.1 - 0.5)
    """
    original_terms = original_query.lower().split()
    # Boost từ gốc cố định để giữ đúng ý định ban đầu (Original Intent)
    q_terms = [f"{t}^2.0" for t in original_terms]

    if not weights:
        return " ".join(q_terms)

    # Lọc bỏ các từ đã có trong query gốc trước khi lấy top_m
    filtered_weights = {t: w for t, w in weights.items() if t not in original_terms}
    
    if not filtered_weights:
        return " ".join(q_terms)

    max_w = max(filtered_weights.values())
    sorted_terms = sorted(filtered_weights.items(), key=lambda x: x[1], reverse=True)[:top_m]

    for term, w in sorted_terms:
        if w <= 0: continue
        
        # Boost cho từ mới = (tỉ lệ so với max) * hệ số tin cậy
        # Điều này đảm bảo từ mới không bao giờ quan trọng bằng từ gốc
        boost = (w / max_w) * expansion_weight_scale
        q_terms.append(f"{term}^{round(boost, 2)}")

    return " ".join(q_terms)

In [28]:
from collections import defaultdict

def bm25_prf_iterative(
    ix_dir,
    queries,
    k1=1.2,
    b=0.75,
    K=10,
    top_m=5,
    max_iter=2
):
    current_queries = queries.copy()
    history = []

    for it in range(max_iter):
        runs, searcher = bm25_retrieve(
            ix_dir,
            current_queries,
            k1=k1,
            b=b,
            top_k=100
        )
        print(f"🔥 Iteration {it+1} done.")
        RD = get_pseudo_relevant(runs, K)

        term_df, R = collect_term_stats(searcher, RD)
        p_terms = estimate_p(term_df, R)
        weights = compute_weights(searcher, p_terms)

        new_queries = {}
        for qid, qtext in current_queries.items():
            new_queries[qid] = expand_query_safe(
                qtext,
                weights,
                top_m=top_m,
                expansion_weight_scale=0.4
            )

        history.append({
            "queries": new_queries,
            "weights": weights
        })

        current_queries = new_queries

    return runs, history


In [29]:

# 2. Run PRF on tuning queries
final_run, history = bm25_prf_iterative(
    ix_dir="ind",
    queries=queries,
    k1=1.2,
    b=0.75,
    K=20,
    top_m=5,
    max_iter=2
)

🔥 Iteration 1 done.
🔥 Iteration 2 done.


In [30]:
queries

{'1': 'nhà_thờ kon_tum',
 '2': 'vũng_tàu địa_điểm đẹp',
 '3': 'địa_điểm du_lịch nổi_tiếng hà_nội',
 '4': 'đi du_lịch đà_nẵng',
 '5': 'du_lịch hội_an trải nghiệm đặc_sắc',
 '6': 'lý_tưởng du_lịch sa_pa',
 '7': 'tham_quan lỡ huế',
 '8': 'du_lịch ninh_bình đi tràng_an tam_cốc',
 '9': 'địa_điểm du_lịch sinh_thái nổi_bật miền tây_nam_bộ',
 '10': 'check in đẹp đà_lạt giới trẻ',
 '11': 'du_lịch hạ_long tour hoạt_động hấp_dẫn',
 '12': 'địa_điểm du_lịch tâm_linh nổi_tiếng việt_nam',
 '13': 'đi du_lịch côn_đảo mùa',
 '14': 'địa_điểm du_lịch tp hcm đi tuần',
 '15': 'du_lịch mộc_châu hấp_dẫn mùa hoa',
 '16': 'vườn quốc_gia đẹp nổi_tiếng việt_nam',
 '17': 'du_lịch quy_nhơn bãi biển hoang_sơ',
 '18': 'du_lịch miền huế đi',
 '19': 'đồng_nai núi',
 '20': 'chùa nổi_tiếng hà_nội',
 '21': 'núi nổi_tiếng leo sa_pa lào_cai',
 '22': 'thác đẹp nổi_tiếng đà_lạt',
 '23': 'hang_động nổi_tiếng quảng_bình',
 '24': 'di_tích lịch_sử nổi_bật cố_đô huế',
 '25': 'cầu nổi_tiếng check in đà_nẵng',
 '26': 'làng_nghề truy

In [ ]:
final_run

{'1': [437,
  57,
  438,
  244,
  259,
  250,
  272,
  313,
  314,
  285,
  409,
  349,
  171,
  196,
  200,
  201,
  221,
  206,
  207,
  258,
  331,
  354,
  319,
  212,
  264,
  214,
  54,
  159,
  271,
  407,
  350,
  66,
  340,
  142,
  359,
  279,
  351,
  161,
  323,
  289,
  304,
  288,
  81,
  398,
  400,
  428,
  431,
  183,
  149,
  341,
  339,
  195,
  80,
  391,
  77,
  166,
  74,
  176,
  310,
  164,
  223,
  389,
  369,
  370,
  399,
  5,
  225,
  33,
  128,
  198,
  83,
  112,
  1,
  16,
  302,
  343,
  94,
  185,
  108,
  346,
  238,
  324,
  61,
  99,
  311,
  133,
  321,
  11,
  240,
  150,
  189,
  37,
  276,
  113,
  266,
  79,
  41,
  91,
  38,
  123],
 '2': [280,
  204,
  421,
  371,
  122,
  373,
  9,
  279,
  205,
  250,
  211,
  370,
  219,
  259,
  324,
  244,
  415,
  331,
  286,
  408,
  283,
  420,
  369,
  332,
  366,
  343,
  317,
  311,
  130,
  393,
  287,
  436,
  435,
  434,
  170,
  141,
  433,
  298,
  194,
  392,
  347,
  368,
  372,
  431,
  321,

In [31]:
from whoosh import index

def runs_docnum_to_eval_format(ix_dir, runs):
    """
    runs: {qid: [docnum, docnum, ...]}
    return: {qid: {docid: score}}
    """
    idx = index.open_dir(ix_dir)
    RunResults = {}

    with idx.searcher() as searcher:
        for qid, docnums in runs.items():
            scores = {}
            n = len(docnums)

            for rank, docnum in enumerate(docnums):
                doc = searcher.stored_fields(docnum)
                docid = str(doc["docid"])

                # score giả theo rank (cao → tốt)
                scores[docid] = float(n - rank)

            RunResults[qid] = scores

    return RunResults


In [32]:
RunResults_for_eval = runs_docnum_to_eval_format(
    ix_dir="ind",
    runs=final_run   # hoặc runs hiện tại của bạn
)

In [33]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    RunResults_for_eval
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.2500
    Recall    : 0.6250
    F1        : 0.3571
------------------------------
Query 3
  @ 5
    Precision : 0.4000
    Recall    : 0.2500
    F1        : 0.3077
  @ 10
    Precision : 0.4000
    Recall    : 0.5000
    F1        : 0.4444
  @ 20
    Precision : 0.2000
    Recall    : 0.5000
    F1        : 0.2857
------------------------------
Query 4
  @ 5
    Precision : 0.8000
    Recall    : 0.3333
    F1        : 0.4706
  @ 10
    Precision : 0.4000
    Recall    : 0.3333
    F1        : 0.3636
  @

## BM25 + KNN + PageRank

In [34]:
def normalize(scores):
    min_s, max_s = min(scores), max(scores)
    return [(s - min_s) / (max_s - min_s + 1e-9) for s in scores]

In [35]:
ALPHA = 0.8

def rank_with_pagerank(run, pagerank_dict):
    """
    run: dict[qid][doc_id] = lsa_score
    pagerank_dict: dict[doc_id] = pagerank
    """
    final_run = {}

    for qid, doc_scores in run.items():
        # lấy danh sách doc_id và score
        doc_ids = list(doc_scores.keys())
        lsa_scores = list(doc_scores.values())

        # lấy pagerank (ép kiểu nếu cần)
        pr_scores = [
            pagerank_dict.get(int(doc_id), 0.0)
            for doc_id in doc_ids
        ]

        # normalize pagerank theo query
        pr_norm = normalize(pr_scores)

        # kết hợp điểm
        reranked = {}
        for doc_id, lsa, pr in zip(doc_ids, lsa_scores, pr_norm):
            score = ALPHA * lsa + (1 - ALPHA) * pr
            reranked[doc_id] = score

        # sort giảm dần
        reranked = dict(
            sorted(reranked.items(), key=lambda x: x[1], reverse=True)
        )

        final_run[qid] = reranked

    return final_run


In [36]:
import pandas as pd

def load_pagerank(meta_csv):
    df = pd.read_csv(meta_csv)

    # giả sử cột là: doc_id, pagerank
    pagerank = dict(zip(df["id"], df["pagerank"]))

    return pagerank

In [ ]:
INDEX_DIR = "ind"
META_CSV = "../../3.AnalysisData/output/final_document_tfidf_pagerank.csv"
JSON_DIR = "../../3.AnalysisData/output/docs_no_stop.json"
QUERY_CSV = "../../3.AnalysisData/test/queries.csv"
# Load PageRank scores
pagerank_dict = load_pagerank(META_CSV)

# 1. Index
ix = build_index(
    index_dir=INDEX_DIR,
    meta_csv=META_CSV,
    json_path=JSON_DIR
)
print("✅ Đã xây dựng xong index")
# 2. Queries + Qrels
queries = readQuery(QUERY_CSV)
qrels = readGroundTruth(QUERY_CSV, META_CSV)
print("Số truy vấn:", len(queries))
# 3. LSA
print("✅ Đã xây dựng xong LSA")
# 4. Run
print("✅ Đã chạy xong tất cả truy vấn")
final_run = rank_with_pagerank(run, pagerank_dict)
print("✅ Đã xếp hạng lại với PageRank")

In [52]:
def normalize(scores):
    min_s, max_s = min(scores), max(scores)
    return [(s - min_s) / (max_s - min_s + 1e-9) for s in scores]

In [53]:
ALPHA = 0.8

def rank_with_pagerank(run, pagerank_dict):
    """
    run: dict[qid][doc_id] = lsa_score
    pagerank_dict: dict[doc_id] = pagerank
    """
    final_run = {}

    for qid, doc_scores in run.items():
        # lấy danh sách doc_id và score
        doc_ids = list(doc_scores.keys())
        lsa_scores = list(doc_scores.values())

        # lấy pagerank (ép kiểu nếu cần)
        pr_scores = [
            pagerank_dict.get(int(doc_id), 0.0)
            for doc_id in doc_ids
        ]

        # normalize pagerank theo query
        pr_norm = normalize(pr_scores)

        # kết hợp điểm
        reranked = {}
        for doc_id, lsa, pr in zip(doc_ids, lsa_scores, pr_norm):
            score = ALPHA * lsa + (1 - ALPHA) * pr
            reranked[doc_id] = score

        # sort giảm dần
        reranked = dict(
            sorted(reranked.items(), key=lambda x: x[1], reverse=True)
        )

        final_run[qid] = reranked

    return final_run


In [50]:
import pandas as pd

def load_pagerank(meta_csv):
    df = pd.read_csv(meta_csv)

    # giả sử cột là: doc_id, pagerank
    pagerank = dict(zip(df["id"], df["pagerank"]))

    return pagerank

In [54]:
# 1. BM25
RunResults_BM25 = run_bm25_all_queries_title(ix, queries, top_k=100)
print("✅ Đã chạy xong tất cả truy vấn BM25")
# Load PageRank scores
pagerank_dict = load_pagerank(META_CSV)
# 2. KNN
docids, X_docs, vectorizer = build_tfidf_vectors(META_CSV, JSON_DIR)
RunResults_KNN = run_knn_all_queries(
    queries, vectorizer, X_docs, docids, top_k=100
)
print("✅ Đã chạy xong tất cả truy vấn KNN")
# 3. Fusion
RunResults_Fused = fuse_runs(
    RunResults_BM25,
    RunResults_KNN,
    alpha=0.6
)
print("✅ Đã fuse xong kết quả BM25 và KNN")
final_run = rank_with_pagerank(RunResults_Fused, pagerank_dict)

print_best_worst_queries(final_run, queries, qrels, top_k=5, retrieved_k=10)

✅ Đã chạy xong tất cả truy vấn BM25


c:\Users\mt200\OneDrive\Desktop\AI\InformationRetrieval\Project_InformationRetrieval\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


✅ Đã chạy xong tất cả truy vấn KNN
✅ Đã fuse xong kết quả BM25 và KNN

🔥 TOP QUERIES (Highest MAP)

Query ID : 11
Query    : du_lịch hạ_long tour hoạt_động hấp_dẫn
MAP      : 1.0000
P@10     : 0.4000
Relevant docs (4): ['50', '122', '132', '133']
Top retrieved docs: ['50', '133', '132', '122', '93', '376', '480', '460', '328', '319']

Query ID : 19
Query    : đồng_nai núi
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['30', '218']
Top retrieved docs: ['30', '218', '168', '186', '331', '169', '377', '433', '188', '432']

Query ID : 20
Query    : chùa nổi_tiếng hà_nội
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['350', '49']
Top retrieved docs: ['350', '49', '444', '301', '18', '17', '388', '390', '401', '403']

Query ID : 30
Query    : ruộng bậc_thang nổi_tiếng miền núi bắc
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['308', '309']
Top retrieved docs: ['308', '309', '188', '70', '86', '126', '111', '100', '62', '107']

Query ID : 33
Query    : đèo đẹp nổi_

In [57]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    final_run
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.3000
    Recall    : 0.3750
    F1        : 0.3333
  @ 20
    Precision : 0.1500
    Recall    : 0.3750
    F1        : 0.2143
------------------------------
Query 4
  @ 5
    Precision : 0.6000
    Recall    : 0.2500
    F1        : 0.3529
  @ 10
    Precision : 0.7000
    Recall    : 0.5833
    F1        : 0.6364
  @

## BM25 + KNN2

### 1. Vector hóa Document (TF-IDF – đơn giản nhất)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

In [61]:
def load_docs(meta_csv, json_path):
    import pandas as pd
    import json
    docs = {}
    df = pd.read_csv(meta_csv)
    with open(json_path, encoding="utf-8") as f:
        doc_terms = json.load(f)

    docids = []
    corpus = []

    for _, row in df.iterrows():
        docid = str(row["id"])
        doc_file = row["document"]
        tokens = doc_terms.get(doc_file, [])
        if not tokens:
            continue

        text = preprocess(tokens)
        docids.append(docid)
        corpus.append(text)
        
        docs[docid] = text
    return docids, corpus, docs

_, _, docs = load_docs(META_CSV, JSON_DIR)

In [64]:
import pickle

vectorizer = TfidfVectorizer()
doc_ids = list(docs.keys())
X = vectorizer.fit_transform(docs.values())

# Lưu lại
pickle.dump((vectorizer, X, doc_ids), open("tfidf.pkl", "wb"))


In [65]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    n_neighbors=50,
    metric="cosine"
)
knn.fit(X)

,n_neighbors,50
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [66]:
def query_vectorize(query, vectorizer):
    return vectorizer.transform([query])

In [67]:
def hybrid_search_doc_knn(
    bm25_results,        # dict {docid: bm25_score} đã sort
    X,                   # document vectors
    doc_ids,             # index -> docid
    knn,                 # fitted NearestNeighbors on X
    top_bm25=50,
    k_knn=10,
    alpha=0.7
):
    scores = {}

    # 1. Top BM25
    bm25_docs = list(bm25_results.items())[:top_bm25]

    for docid, bm25_score in bm25_docs:
        scores[docid] = alpha * bm25_score

    # map docid -> index
    docid_to_idx = {d: i for i, d in enumerate(doc_ids)}

    # 2. Doc–doc KNN expansion
    for docid, bm25_score in bm25_docs:
        if docid not in docid_to_idx:
            continue

        doc_idx = docid_to_idx[docid]
        doc_vec = X[doc_idx].reshape(1, -1)

        dist, idxs = knn.kneighbors(doc_vec, n_neighbors=k_knn)

        for d, i in zip(dist[0], idxs[0]):
            neighbor_docid = doc_ids[i]
            sim = 1 - d  # cosine similarity

            scores[neighbor_docid] = scores.get(neighbor_docid, 0) + \
                                     (1 - alpha) * sim

    # 3. Sort
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

In [78]:
def compute_P_at_10(GroundTruth, RunResults):
    evaluator = pytrec_eval.RelevanceEvaluator(
        GroundTruth, {"P_10"}
    )
    results = evaluator.evaluate(RunResults)

    P10 = 0.0
    valid_queries = 0

    for qid, res in results.items():
        if not math.isnan(res["P_10"]):
            P10 += res["P_10"]
            valid_queries += 1

    return P10 / valid_queries if valid_queries > 0 else 0.0


In [69]:
import optuna

c:\Users\mt200\OneDrive\Desktop\AI\InformationRetrieval\Project_InformationRetrieval\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [79]:
def objective(trial):
    # 🔧 Siêu tham số cần tối ưu
    top_bm25 = trial.suggest_int("top_bm25", 20, 100)
    k_knn = trial.suggest_int("k_knn", 5, 50)
    alpha = trial.suggest_float("alpha", 0.0, 1.0)

    RunResults_Fused = {}

    for qid, qtext in queries.items():
        bm25_res = final_run[qid]

        fused = hybrid_search_doc_knn(
            bm25_res,
            X,
            doc_ids,
            knn,
            top_bm25=top_bm25,
            k_knn=k_knn,
            alpha=alpha
        )

        RunResults_Fused[qid] = {
            docid: float(score)
            for docid, score in fused.items()
        }

    # 🎯 Objective = P@10
    return compute_P_at_10(qrels, RunResults_Fused)


In [80]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(study.best_params)
print(study.best_value)


[I 2025-12-30 09:05:17,804] A new study created in memory with name: no-name-8b90100f-3e88-40fd-ac96-2fd504dc3552
[I 2025-12-30 09:06:07,046] Trial 0 finished with value: 0.03 and parameters: {'top_bm25': 96, 'k_knn': 10, 'alpha': 0.0074893390710278895}. Best is trial 0 with value: 0.03.
[I 2025-12-30 09:06:36,571] Trial 1 finished with value: 0.05250000000000001 and parameters: {'top_bm25': 63, 'k_knn': 12, 'alpha': 0.16922107674745912}. Best is trial 1 with value: 0.05250000000000001.
[I 2025-12-30 09:07:05,261] Trial 2 finished with value: 0.07750000000000004 and parameters: {'top_bm25': 63, 'k_knn': 28, 'alpha': 0.7341841804247999}. Best is trial 2 with value: 0.07750000000000004.
[I 2025-12-30 09:07:35,257] Trial 3 finished with value: 0.047500000000000014 and parameters: {'top_bm25': 74, 'k_knn': 47, 'alpha': 0.5917366625000409}. Best is trial 2 with value: 0.07750000000000004.
[I 2025-12-30 09:07:55,267] Trial 4 finished with value: 0.06000000000000001 and parameters: {'top_bm25

{'top_bm25': 38, 'k_knn': 45, 'alpha': 0.9968660417250852}
0.24999999999999994


In [82]:
best_params = study.best_params
best_map = study.best_value

print("Best MAP:", best_map)
print("Best params:", best_params)


Best MAP: 0.24999999999999994
Best params: {'top_bm25': 38, 'k_knn': 45, 'alpha': 0.9968660417250852}


In [83]:
RunResults_Best = {}

for qid, qtext in queries.items():
    fused = hybrid_search_doc_knn(
        final_run[qid],
        X,
        doc_ids,
        knn,
        top_bm25=best_params["top_bm25"],
        k_knn=best_params["k_knn"],
        alpha=best_params["alpha"]
    )

    RunResults_Best[qid] = {
        docid: float(score)
        for docid, score in fused.items()
    }


In [86]:
per_query, avg = evaluate_set_retrieval_at_k(
    qrels,
    RunResults_Best
)

========== Per-query results ==========
Query 1
  @ 5
    Precision : 0.4000
    Recall    : 1.0000
    F1        : 0.5714
  @ 10
    Precision : 0.2000
    Recall    : 1.0000
    F1        : 0.3333
  @ 20
    Precision : 0.1000
    Recall    : 1.0000
    F1        : 0.1818
------------------------------
Query 2
  @ 5
    Precision : 0.8000
    Recall    : 0.5000
    F1        : 0.6154
  @ 10
    Precision : 0.5000
    Recall    : 0.6250
    F1        : 0.5556
  @ 20
    Precision : 0.3000
    Recall    : 0.7500
    F1        : 0.4286
------------------------------
Query 3
  @ 5
    Precision : 0.6000
    Recall    : 0.3750
    F1        : 0.4615
  @ 10
    Precision : 0.3000
    Recall    : 0.3750
    F1        : 0.3333
  @ 20
    Precision : 0.1500
    Recall    : 0.3750
    F1        : 0.2143
------------------------------
Query 4
  @ 5
    Precision : 0.6000
    Recall    : 0.2500
    F1        : 0.3529
  @ 10
    Precision : 0.7000
    Recall    : 0.5833
    F1        : 0.6364
  @

In [87]:
print_best_worst_queries(RunResults_Best, queries, qrels, top_k=5, retrieved_k=10)


🔥 TOP QUERIES (Highest MAP)

Query ID : 11
Query    : du_lịch hạ_long tour hoạt_động hấp_dẫn
MAP      : 1.0000
P@10     : 0.4000
Relevant docs (4): ['50', '122', '132', '133']
Top retrieved docs: ['50', '133', '132', '122', '93', '376', '480', '460', '328', '95']

Query ID : 19
Query    : đồng_nai núi
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['30', '218']
Top retrieved docs: ['30', '218', '168', '186', '331', '169', '377', '433', '188', '149']

Query ID : 20
Query    : chùa nổi_tiếng hà_nội
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['350', '49']
Top retrieved docs: ['350', '49', '301', '444', '18', '17', '388', '390', '401', '403']

Query ID : 30
Query    : ruộng bậc_thang nổi_tiếng miền núi bắc
MAP      : 1.0000
P@10     : 0.2000
Relevant docs (2): ['308', '309']
Top retrieved docs: ['308', '309', '70', '188', '126', '86', '111', '100', '62', '107']

Query ID : 33
Query    : đèo đẹp nổi_tiếng phượt miền trung
MAP      : 1.0000
P@10     : 0.2000
Relevant doc